# 31. SegFormer 내부계측 파이프라인

Chapter 2-2에서 학습한 SegFormer seed 반복 모델을 재사용해 Chapter 3 실험의 공통 기반을 만듭니다.

이 노트북의 산출물:

- `runs/ch3_model_registry.csv`
- `runs/manifests/standard/*`
- `runs/feature_bank/.../stage_features.npz`
- `runs/feature_bank/.../sample_index.csv`
- `runs/feature_bank/.../stage_energy.csv`

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch3_utils.py").exists():
    matches = list(Path.cwd().glob("Deeplearning/*/3장/ch3_utils.py")) + list(Path.cwd().glob("**/ch3_utils.py"))
    if matches:
        NOTEBOOK_DIR = matches[0].parent
    else:
        NOTEBOOK_DIR = Path("Deeplearning") / "Vision 응용" / "3장"
sys.path.insert(0, str(NOTEBOOK_DIR))

from ch3_utils import *

paths = find_ch3_paths()
set_korean_font()
set_seed(31)
paths

Chapter3Paths(chapter3_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장'), chapter2_2_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장'), data_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/data'), stress_ladder_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/data/synthetic_metal_stress_ladder'), runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/runs'), manifest_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/runs/manifests'), ch2_2_runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장/runs'))

## 31-1. Chapter 2-2 산출물 확인

In [2]:
required = require_ch2_2_outputs()
for name, path in required.items():
    print(f"{name:20s}", path)

samples = load_ch3_base_samples()
print("samples:", len(samples))
display(samples.groupby(["split", "color_group", "defect_type"]).size().reset_index(name="count").head(20))

samples              C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\data\synthetic_metal_matched\metadata\samples.csv
audit                C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\runs\audit\audit_summary.json
baseline_summary     C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\runs\baseline_seed_summary.csv
factor_tests         C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\runs\matched_factor_tests.csv
hypothesis_validity  C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\runs\chapter2_2_hypothesis_validity_report.md
samples: 736


,split,color_group,defect_type,count
0,eval_matched,blue,dent,12
1,eval_matched,blue,impact,12
2,eval_matched,blue,scratch,12
3,eval_matched,blue,stain,12
4,eval_matched,green,dent,12
5,eval_matched,green,impact,12
6,eval_matched,green,scratch,12
7,eval_matched,green,stain,12
8,eval_matched,neutral,dent,12
9,eval_matched,neutral,impact,12


## 31-2. Chapter 3 probe manifest와 모델 registry 생성

In [3]:
# max_per_cell=None이면 eval_matched 전체를 사용합니다.
# 빠른 계측 smoke run은 2~3으로 두고, 최종 산출물은 None 또는 6을 권장합니다.
PROBE_MAX_PER_CELL = None

manifests = create_ch3_probe_manifests(samples, max_per_cell=PROBE_MAX_PER_CELL, seed=31)
registry = discover_ch3_model_registry()

for name, path in manifests.items():
    print(f"{name:32s}", path, "rows=", len(pd.read_csv(path)))
display(registry)

train                            C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\manifests\standard\train_manifest.csv rows= 288
eval_matched                     C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\manifests\standard\eval_matched_manifest.csv rows= 240
eval_stress                      C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\manifests\standard\eval_stress_manifest.csv rows= 160
eval_matched_probe               C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\manifests\standard\eval_matched_probe_manifest.csv rows= 240
eval_color_counterfactual_probe  C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\manifests\standard\eval_color_counterfactual_probe_manifest.csv rows= 192


,model_key,family,variant,model_seed,run_dir,checkpoint_exists,config_exists,hf_config_exists
0,baseline/baseline_no_aug/seed_0,baseline,baseline_no_aug,0,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,True,True,True
1,baseline/baseline_no_aug/seed_1,baseline,baseline_no_aug,1,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,True,True,True
2,baseline/baseline_no_aug/seed_2,baseline,baseline_no_aug,2,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,True,True,True
3,strategy/group_balanced/seed_0,strategy,group_balanced,0,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,True,True,True
4,strategy/group_balanced/seed_1,strategy,group_balanced,1,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,True,True,True
5,strategy/group_balanced/seed_2,strategy,group_balanced,2,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,True,True,True
6,strategy/photometric_aug/seed_0,strategy,photometric_aug,0,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,True,True,True
7,strategy/photometric_aug/seed_1,strategy,photometric_aug,1,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,True,True,True
8,strategy/photometric_aug/seed_2,strategy,photometric_aug,2,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,True,True,True
9,exposure_ratio/red_scratch_ratio_0p00/seed_0,exposure_ratio,red_scratch_ratio_0p00,0,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,True,True,True


## 31-3. baseline seed 0 feature bank 추출

In [4]:
MODEL_FAMILY = "baseline"
MODEL_VARIANT = "baseline_no_aug"
MODEL_SEED = 0
FEATURE_MAX_SAMPLES = None  # None이면 manifest 전체 사용
BATCH_SIZE = 8

selected = registry[
    (registry["family"] == MODEL_FAMILY)
    & (registry["variant"] == MODEL_VARIANT)
    & (registry["model_seed"] == MODEL_SEED)
    & (registry["checkpoint_exists"])
]
if selected.empty:
    raise FileNotFoundError("선택한 모델 checkpoint를 찾지 못했습니다. 2-2장의 24번 노트북을 먼저 실행하세요.")

model_run_dir = Path(selected.iloc[0]["run_dir"])
feature_out = paths.runs_root / "feature_bank" / MODEL_VARIANT / f"seed_{MODEL_SEED}"
feature_paths = extract_feature_bank(
    model_run_dir,
    manifests["eval_matched_probe"],
    feature_out,
    batch_size=BATCH_SIZE,
    max_samples=FEATURE_MAX_SAMPLES,
    seed=31,
)
feature_paths

C:\Users\준승\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'features': WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/runs/feature_bank/baseline_no_aug/seed_0/stage_features.npz'),
 'index': WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/runs/feature_bank/baseline_no_aug/seed_0/sample_index.csv'),
 'energy': WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/runs/feature_bank/baseline_no_aug/seed_0/stage_energy.csv')}

## 31-4. 여러 모델 feature bank 추출

In [5]:
RUN_ALL_BASELINE_SEEDS = True
RUN_PHOTOMETRIC_MODEL = True

jobs = []
if RUN_ALL_BASELINE_SEEDS:
    jobs.extend(
        registry[
            (registry["family"] == "baseline")
            & (registry["variant"] == "baseline_no_aug")
            & (registry["checkpoint_exists"])
        ].to_dict(orient="records")
    )
if RUN_PHOTOMETRIC_MODEL:
    jobs.extend(
        registry[
            (registry["family"] == "strategy")
            & (registry["variant"] == "photometric_aug")
            & (registry["checkpoint_exists"])
        ].to_dict(orient="records")
    )

for job in jobs:
    out_dir = paths.runs_root / "feature_bank" / job["variant"] / f"seed_{int(job['model_seed'])}"
    print("extracting:", job["model_key"])
    extract_feature_bank(
        job["run_dir"],
        manifests["eval_matched_probe"],
        out_dir,
        batch_size=BATCH_SIZE,
        max_samples=FEATURE_MAX_SAMPLES,
        seed=31,
    )

extracting: baseline/baseline_no_aug/seed_0


extracting: baseline/baseline_no_aug/seed_1


extracting: baseline/baseline_no_aug/seed_2


extracting: strategy/photometric_aug/seed_0


extracting: strategy/photometric_aug/seed_1


extracting: strategy/photometric_aug/seed_2
